# 03 · Blend and final submission

Pick runs by **CV**, check they are diverse (correlation < ~0.95), blend on OOF, and keep the
final choice robust: prefer the blend whose CV is best *and* stable across folds over the one
with the best public score (public LB on a few thousand rows is noise at ±0.3–1 pt).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
# os.environ["QRTC_CONFIG"] = "experiments/intraday_ridge/spec.toml"  # another spec
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60, "display.width", 160)
from qrtc import config, io, eda, features as F, cv, train, tracking, ensemble, postprocess, submit
from qrtc.metrics import get_metric
spec = config.load_spec()
spec

In [ ]:
ds = io.load_dataset(spec)
lb = tracking.leaderboard(competition=spec.name)
lb.head(20)

In [ ]:
RUNS = lb.loc[lb["local"], "run_id"].head(5).tolist()   # edit: hand-pick diverse, strong runs
ids = ds.train[spec.id_col].to_numpy(), ds.test[spec.id_col].to_numpy()
preds = {r: tracking.load_preds(r, *ids) for r in RUNS}   # refuses runs made on other data
ensemble.correlation_table({r: p["oof"] for r, p in preds.items()})

In [ ]:
metric = get_metric(spec.metric)
groups = ds.train[spec.date_col].to_numpy() if spec.date_col else None
weight = io.train_weight(ds, spec)
oofs, tests = [preds[r]["oof"] for r in RUNS], [preds[r]["test"] for r in RUNS]
rows = ensemble.common_rows(oofs)
rank = spec.metric in ("spearman", "mean_spearman", "auc", "cindex", "ipcw_cindex")
scorer = lambda p: train.score_predictions(spec, ds.y_vector, p, rows, weight, groups)
for r, o in zip(RUNS, oofs):
    print(f"{scorer(o):.5f}  {r}")
w = ensemble.hill_climb(oofs, scorer, metric.greater_is_better, rank=rank)
blend_oof, blend_test = ensemble.weighted_blend(oofs, w, rank), ensemble.weighted_blend(tests, w, rank)
print("weights", w, "blend OOF", scorer(blend_oof))

## Post-processing for label metrics (accuracy): threshold vs per-date median split

In [ ]:
threshold = 0.5
if spec.task == "binary" and spec.metric in submit.LABEL_METRICS:
    threshold, s = postprocess.best_threshold(ds.y_vector[rows], blend_oof[rows], metric)
    print("best OOF threshold", threshold, s, "(only trust it if close to 0.5 or very stable)")
    if groups is not None:
        med = postprocess.median_split_within(blend_oof[rows], groups[rows])
        print("per-date median split OOF:", metric(ds.y_vector[rows], med))

## Write, validate, record

In [ ]:
test_ids = ds.test[spec.id_col].to_numpy()
res = train.CVResult(model="blend", params={"runs": RUNS, "w": list(map(float, w))}, features=(),
                     metric=spec.metric, score=scorer(blend_oof), score_std=0.0,
                     fold_scores=(scorer(blend_oof),), oof=blend_oof, test_pred=blend_test, elapsed_s=0)
run_id = tracking.save_run(res, spec, "final-blend", ds.train[spec.id_col].to_numpy(), test_ids)
sub = submit.build_submission(spec, test_ids, blend_test, ds.classes, threshold=threshold)
print(submit.write_submission(sub, spec, test_ids, run_id))